# Lab 44 — Modelo de Hubbard y VQE+DMRG

El modelo de Hubbard es uno de los Hamiltonianos más estudiados en física de materia condensada. Captura la competición entre hopping electrónico y repulsión de Coulomb en sitios de red.

$$H = -t\sum_{\langle i,j\rangle, \sigma} (c_{i\sigma}^\dagger c_{j\sigma} + \text{h.c.}) + U\sum_i n_{i\uparrow} n_{i\downarrow}$$

**Contenido:**
- Parte A: Mapeo de Jordan-Wigner al espacio de qubits
- Parte B: VQE para la cadena de Hubbard de 4 sitios
- Parte C: Comparativa VQE vs DMRG (Matrix Product States)
- Parte D: Diagrama de fase t-U y transición metal-aislante

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

print('Entorno listo para modelo de Hubbard')

## Parte A: Mapeo de Jordan-Wigner

Para mapear operadores fermiónicos a operadores de espín usamos el mapeo de Jordan-Wigner:

$$c_j^\dagger = \left(\prod_{k<j} Z_k\right) \cdot \frac{X_j - iY_j}{2}$$

$$c_j = \left(\prod_{k<j} Z_k\right) \cdot \frac{X_j + iY_j}{2}$$

Para un modelo de Hubbard de $L$ sitios con 2 spines: $2L$ qubits (L sitios × 2 espines).

In [ ]:
def hubbard_hamiltonian_jw(L: int, t: float, U: float) -> SparsePauliOp:
    """Hamiltoniano de Hubbard 1D mediante Jordan-Wigner para L sitios, 2L qubits.

    Ordenamiento: qubits 0..L-1 = espín up, L..2L-1 = espín down.
    Término de hopping: -t(c_i↑† c_{i+1}↑ + h.c.) y lo mismo para down.
    Término de interacción: U n_{i↑} n_{i↓} = U(I-Z_i↑)/2 · (I-Z_{i↓})/2
    """
    n_qubits = 2 * L
    terms = []

    # Hopping spin-up (sitios 0..L-1)
    for i in range(L - 1):
        # -t/2 (X_i Z_{i+1...} X_{i+1} + Y_i Z_{i+1...} Y_{i+1})
        # Para vecinos inmediatos sin cadena Z intermedia:
        xx_op = ['I'] * n_qubits
        xx_op[i] = 'X'; xx_op[i+1] = 'X'
        yy_op = ['I'] * n_qubits
        yy_op[i] = 'Y'; yy_op[i+1] = 'Y'
        terms.append((''.join(reversed(xx_op)), -t / 2))
        terms.append((''.join(reversed(yy_op)), -t / 2))

    # Hopping spin-down (sitios L..2L-1)
    for i in range(L - 1):
        xx_op = ['I'] * n_qubits
        xx_op[L+i] = 'X'; xx_op[L+i+1] = 'X'
        yy_op = ['I'] * n_qubits
        yy_op[L+i] = 'Y'; yy_op[L+i+1] = 'Y'
        terms.append((''.join(reversed(xx_op)), -t / 2))
        terms.append((''.join(reversed(yy_op)), -t / 2))

    # Interacción U: n_{i↑}n_{i↓} = (I - Z_{i↑})(I - Z_{i↓})/4
    for i in range(L):
        # Constante: U/4
        terms.append(('I' * n_qubits, U / 4))
        # -U/4 Z_{i↑}
        z_up = ['I'] * n_qubits
        z_up[i] = 'Z'
        terms.append((''.join(reversed(z_up)), -U / 4))
        # -U/4 Z_{i↓}
        z_dn = ['I'] * n_qubits
        z_dn[L+i] = 'Z'
        terms.append((''.join(reversed(z_dn)), -U / 4))
        # +U/4 Z_{i↑}Z_{i↓}
        zz_op = ['I'] * n_qubits
        zz_op[i] = 'Z'; zz_op[L+i] = 'Z'
        terms.append((''.join(reversed(zz_op)), U / 4))

    return SparsePauliOp.from_list(terms, num_qubits=n_qubits).simplify()


# Construir Hamiltoniano para L=2 sitios (4 qubits) como prueba
L = 2
t_val = 1.0
U_val = 4.0
H_hub = hubbard_hamiltonian_jw(L, t_val, U_val)
print(f'Hubbard L={L}: {H_hub.num_qubits} qubits, {len(H_hub)} términos de Pauli')

# Energía exacta por diagonalización
H_matrix = H_hub.to_matrix()
energies, _ = eigh(H_matrix)
E0_exact = energies[0]
print(f'Energía del estado fundamental exacta E0 = {E0_exact:.6f}')
print(f'Primeros 4 niveles: {energies[:4]}')

## Parte B: VQE para Hubbard L=2

In [ ]:
def hubbard_ansatz(n_qubits: int, n_layers: int) -> QuantumCircuit:
    """Ansatz adaptado para Hubbard: simetría partícula-hueco."""
    params = ParameterVector('θ', n_qubits * n_layers * 2)
    qc = QuantumCircuit(n_qubits)

    # Estado inicial half-filling: |↑↓00⟩ (un electrón por espín)
    qc.x(0)  # spin-up sitio 0
    qc.x(n_qubits // 2)  # spin-down sitio 0

    idx = 0
    for _ in range(n_layers):
        for q in range(n_qubits):
            qc.ry(params[idx], q)
            qc.rz(params[idx + 1], q)
            idx += 2
        # Entanglement que respeta la estructura de hopping
        for q in range(0, n_qubits - 1, 2):
            qc.cx(q, q + 1)

    return qc


n_qubits = 2 * L
qc_ansatz = hubbard_ansatz(n_qubits, n_layers=2)
print(f'Ansatz: {qc_ansatz.num_parameters} parámetros, {n_qubits} qubits')

# VQE
estimator = StatevectorEstimator()
energy_history = []

def cost_fn(params_val: np.ndarray) -> float:
    job = estimator.run([(qc_ansatz, H_hub, [params_val])])
    E = float(job.result()[0].data.evs)
    energy_history.append(E)
    return E


np.random.seed(0)
theta0 = np.random.uniform(-np.pi, np.pi, qc_ansatz.num_parameters)
result = minimize(cost_fn, theta0, method='COBYLA',
                  options={'maxiter': 400, 'rhobeg': 0.3})

E_vqe = result.fun
error = abs(E_vqe - E0_exact)

print(f'\nResultados VQE:')
print(f'  E_VQE    = {E_vqe:.6f}')
print(f'  E_exacta = {E0_exact:.6f}')
print(f'  Error    = {error:.2e} ({100*error/abs(E0_exact):.3f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energy_history, 'b-', lw=1.2, alpha=0.8, label='VQE energía')
ax.axhline(E0_exact, color='red', ls='--', lw=2, label=f'E₀ exacta = {E0_exact:.4f}')
ax.set_xlabel('Iteración')
ax.set_ylabel('Energía (u.a.)')
ax.set_title(f'VQE — Modelo de Hubbard L={L}, t={t_val}, U={U_val}')
ax.legend()
plt.tight_layout()
plt.show()

## Parte C: Comparativa VQE vs DMRG (simulado con MPS)

DMRG (Density Matrix Renormalization Group) usa Matrix Product States (MPS) para simular sistemas 1D eficientemente. Para una cadena de Hubbard de L sitios con dimensión de enlace χ:

- Complejidad DMRG: O(χ³ · L) por barrido
- Para estados con bajo entrelazamiento: χ pequeño → exacto eficientemente
- En el punto crítico U/t ~ 4-8: entrelazamiento crece → χ grande

A continuación simulamos el espectro DMRG usando diagonalización exacta (accesible para L≤4).

In [ ]:
def exact_hubbard_energy(L: int, t: float, U: float) -> float:
    """Energía exacta del estado fundamental de Hubbard-1D via diagonalización."""
    H = hubbard_hamiltonian_jw(L, t, U)
    H_mat = H.to_matrix()
    energies = eigh(H_mat, eigvals_only=True)
    return float(energies[0])


# Escanear U/t para L=2
U_over_t = np.linspace(0, 16, 25)
E0_scan = [exact_hubbard_energy(2, 1.0, U) for U in U_over_t]

# Límite t=0 (sin hopping): E_Mott = 0 (sin doble ocupación en half-filling)
# Límite U=0 (sin interacción): E_metal = -2t (banda llena half-filling L=2)
E_metal = [-2.0] * len(U_over_t)  # hopping puro

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(U_over_t, E0_scan, 'b-o', ms=5, lw=2, label='E₀ exacta (ED)')
ax.axvline(4, color='red', ls='--', alpha=0.7, label='U/t ≈ 4 (transición Mott)')
ax.set_xlabel('U/t')
ax.set_ylabel('Energía del estado fundamental E₀/t')
ax.set_title('Hubbard 1D (L=2): Energía vs interacción U/t')
ax.legend()
plt.tight_layout()
plt.show()

print('En U/t → 0: sistema metálico (hopping domina)')
print('En U/t → ∞: aislante de Mott (repulsión Coulomb domina)')

## Parte D: Diagrama de Fase y Transición Metal-Aislante

In [ ]:
# Gap de carga (gap de excitación de partícula) como indicador de la transición
def charge_gap(L: int, t: float, U: float) -> float:
    """Gap de carga = E₀(N+1) + E₀(N-1) - 2E₀(N) — indicador de transición Mott."""
    # Para L=2 half-filling N=2 (1 up, 1 down): calcular energías de los sectores N=1,2,3
    H = hubbard_hamiltonian_jw(L, t, U)
    H_mat = H.to_matrix()
    all_E = np.sort(np.real(np.linalg.eigvalsh(H_mat)))
    # Aproximar el gap con diferencias de energía del espectro
    # (sin particle-number symmetry exacta aquí)
    gap = all_E[2] - all_E[0]  # excitación de 2 niveles
    return max(0.0, gap)


U_scan = np.linspace(0.1, 20, 30)
gaps = [charge_gap(2, 1.0, U) for U in U_scan]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gap vs U
axes[0].plot(U_scan, gaps, 'g-o', ms=4, lw=2)
axes[0].axvline(4, color='red', ls='--', label='U/t ≈ 4')
axes[0].set_xlabel('U/t')
axes[0].set_ylabel('Gap de excitación (u.a.)')
axes[0].set_title('Gap de carga: apertura en transición Mott')
axes[0].legend()

# Diagrama de fase esquemático t-U
t_vals = np.linspace(0.1, 3, 50)
U_critical = 4 * t_vals  # U_c ≈ 4t en aprox. Gutzwiller

axes[1].plot(t_vals, U_critical, 'r-', lw=2, label='Frontera Mott U_c ≈ 4t')
axes[1].fill_between(t_vals, 0, U_critical, alpha=0.2, color='blue', label='Metal (Fermi liquid)')
axes[1].fill_between(t_vals, U_critical, max(U_critical) * 1.5,
                     alpha=0.2, color='red', label='Aislante Mott')
axes[1].set_xlabel('t (hopping)')
axes[1].set_ylabel('U (repulsión)')
axes[1].set_title('Diagrama de fase esquemático Hubbard 1D')
axes[1].set_xlim(0, 3)
axes[1].set_ylim(0, 16)
axes[1].legend()

plt.suptitle('Transición Metal-Aislante de Mott en el modelo de Hubbard')
plt.tight_layout()
plt.show()

## Resumen

| Método | Sistema | Complejidad | Error típico |
|--------|---------|-------------|-------------|
| Diagonalización exacta (ED) | L ≤ 20 qubits | O(2^(2L)) | Exacto |
| DMRG / MPS | L ≤ 1000, bajo entrelazamiento | O(χ³L) | < 10⁻⁶ |
| VQE (NISQ) | L ≤ 20, profundidad limitada | O(shots·iter) | ~10⁻³–10⁻² |
| VQE fault-tolerant | L > 100 (futuro) | Polinomial | ~10⁻⁶ |

**Por qué Hubbard importa:**
- Modelo minimal de superconductores de alta temperatura (cupratos)
- Benchmark estándar para algoritmos cuánticos de química/materiales
- La región de acoplamiento intermedio U/t ≈ 1–8 desafía todos los métodos clásicos
- Es el caso de uso más citado para ventaja cuántica práctica en química

**Siguientes pasos:**
- Implementar con `qiskit-nature` y fermionic mappers (JW, Bravyi-Kitaev)
- Escalar a L=4 con ansatz UCCSD adaptado
- Explorar el punto cuántico crítico con mediciones de entropía de entrelazamiento